# 12 — Eastern Lea County, NM: Multi-Source Ensemble Time Series (2020–2022)

Comprehensive field boundary delineation using **all available satellite products and engines** with vote-based ensemble merging. Grand ensemble boundaries are refined by **SAM2** after majority-vote merging.

**Sources:** Sentinel-2, Landsat, HLS, NAIP, SPOT, Google & TESSERA embeddings
**Engines:** delineate-anything, FTW, GeoAI, DINOv3, Prithvi, embedding

**Study area:** Eastern Lea County (~20×22 km bbox over center pivot area).

**Estimated runtime:** ~2–4 hours (3 years × up to 20+ source–engine–model combos, GPU recommended).

**Prerequisites:**
```bash
pip install agribound[gee,delineate-anything,ftw,geoai,prithvi,samgeo]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
import json
import logging
from pathlib import Path

import geopandas as gpd

import agribound
from agribound.evaluate import evaluate

# Enable agribound logging so download/processing progress is visible
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(message)s",
    datefmt="%H:%M:%S",
)

GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

NMOSE_SHAPEFILE = "../NMOSE Field Boundaries/WUCB ag polys.shp"
OUTPUT_DIR = Path("../../outputs/lea_county_ensemble")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CRS = "EPSG:26913"  # Match NMOSE reference CRS (NAD83 / UTM zone 13N)

COUNTY_CODE = "25"  # Lea County
FINE_TUNE = True  # Fine-tune engines on NMOSE reference boundaries
FINE_TUNE_EPOCHS = 10  # Set to 20 for production runs
FINE_TUNE_ENGINES = {"delineate-anything", "geoai", "prithvi", "dinov3"}
BATCH_SIZE = 8  # Increase for more RAM (e.g. 16 for 128 GB)
SAM_REFINE = True  # Refine grand ensemble boundaries with SAM2
SAM_MODEL = "tiny"  # SAM2 variant: "tiny", "small", "base_plus", "large"
SAM_BATCH_SIZE = 50  # Number of field boxes per SAM2 batch
YEARS = range(2020, 2023)
VOTE_THRESHOLD = 0.3  # Fraction of source–engine combos that must agree

# Source → compatible engines
# For "ftw" entries, each FTW model is run separately via FTW_MODELS below.
SOURCE_ENGINE_MAP = {
    "sentinel2": ["ftw", "geoai", "dinov3", "prithvi", "delineate-anything"],
    "landsat": ["ftw", "dinov3", "prithvi", "delineate-anything"],
    "hls": ["ftw", "dinov3", "prithvi", "delineate-anything"],
    "naip": ["geoai", "dinov3", "delineate-anything"],
    "spot": ["dinov3", "delineate-anything"],
    "google-embedding": ["embedding"],
    "tessera-embedding": ["embedding"],
}

# FTW v3 EfficientNet models — each is run separately for ensemble diversity.
FTW_MODELS = ["FTW_PRUE_EFNET_B3", "FTW_PRUE_EFNET_B5", "FTW_PRUE_EFNET_B7"]

# Delineate-Anything model variants — both are run.
DA_MODELS = ["DelineateAnything", "DelineateAnything-S"]

# Year availability constraints
SOURCE_YEAR_RANGE = {
    "sentinel2": (2017, 2025),
    "landsat": (1985, 2025),
    "hls": (2013, 2025),
    "naip": (2003, 2025),
    "spot": (2012, 2023),
    "google-embedding": (2018, 2024),
    "tessera-embedding": (2017, 2024),
}

# Save reference boundaries for fine-tuning
ref_path = str(OUTPUT_DIR / "lea_county_reference.gpkg")

## Derive Lea County Study Area and Load Reference Boundaries

In [ ]:
from shapely.geometry import box

full_gdf = gpd.read_file(NMOSE_SHAPEFILE)
county_gdf = full_gdf[full_gdf["County"] == COUNTY_CODE].copy()

# Eastern Lea County bbox (center pivot area) — same as example 14
bbox_geojson = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [-103.25, 32.75],
                        [-103.05, 32.75],
                        [-103.05, 32.95],
                        [-103.25, 32.95],
                        [-103.25, 32.75],
                    ]
                ],
            },
            "properties": {"name": f"Eastern Lea County (County {COUNTY_CODE})"},
        }
    ],
}
study_area = str(OUTPUT_DIR / "lea_county_study_area.geojson")
with open(study_area, "w") as f:
    json.dump(bbox_geojson, f)

# Clip reference boundaries to the study area bbox
bbox_geom = box(-103.25, 32.75, -103.05, 32.95)
county_4326 = county_gdf.to_crs(epsg=4326)
ref_gdf = county_4326[county_4326.intersects(bbox_geom)].copy()
ref_gdf = ref_gdf.to_crs(county_gdf.crs)

ref_path_obj = Path(ref_path)
if not ref_path_obj.exists():
    ref_gdf.to_file(ref_path, driver="GPKG")

print(f"Eastern Lea County: {len(ref_gdf)} reference polygons")
if FINE_TUNE:
    print(f"Fine-tuning enabled ({FINE_TUNE_EPOCHS} epochs) using: {ref_path}")

## Phase 1: Per-Source, Per-Engine Delineation

Run every compatible source–engine combination individually for each year.

In [ ]:
def run_delineation(source, engine, year, model=None):
    """Run a single source–engine delineation, returning (GeoDataFrame, path)."""
    suffix = f"_{model}" if model else ""
    output_path = OUTPUT_DIR / f"fields_{source}_{engine}{suffix}_{year}.gpkg"

    if output_path.exists():
        return gpd.read_file(output_path), output_path

    kwargs = dict(
        study_area=study_area,
        source=source,
        year=year,
        engine=engine,
        output_path=str(output_path),
        gee_project=GEE_PROJECT,
        min_area=2500,
        simplify=2.0,
        device="auto",
        engine_params={"batch_size": BATCH_SIZE},
    )

    # Fine-tune on NMOSE reference boundaries if supported
    if FINE_TUNE and engine in FINE_TUNE_ENGINES:
        kwargs["reference_boundaries"] = ref_path
        kwargs["fine_tune"] = True
        kwargs["fine_tune_epochs"] = FINE_TUNE_EPOCHS

    # Source-specific composite parameters
    if source in ("sentinel2", "landsat", "hls"):
        kwargs["composite_method"] = "median"
        kwargs["cloud_cover_max"] = 20
        if engine != "ftw":
            kwargs["date_range"] = (f"{year}-10-01", f"{year}-10-31")
    elif source == "spot":
        kwargs["composite_method"] = "median"
        kwargs["cloud_cover_max"] = 15
    elif source == "naip":
        kwargs["min_area"] = 5000
    elif source in ("google-embedding", "tessera-embedding"):
        kwargs["device"] = "cpu"
        kwargs["min_area"] = 5000
        kwargs["engine_params"].update(
            {
                "use_pca": True,
                "pca_components": 16,
                "n_clusters": "auto",
            }
        )

    # Model override for FTW or Delineate-Anything
    if model and engine == "ftw":
        kwargs["engine_params"]["model"] = model
    elif model and engine == "delineate-anything":
        kwargs["engine_params"]["da_model"] = model

    gdf = agribound.delineate(**kwargs)
    # Reproject to match NMOSE reference CRS
    if gdf.crs is not None and str(gdf.crs) != OUTPUT_CRS:
        gdf = gdf.to_crs(OUTPUT_CRS)
        gdf.to_file(output_path, driver="GPKG")
    return gdf, output_path


all_results = {}  # {year: {"source/engine": gdf}}

for year in YEARS:
    print(f"\n--- Year {year} ---")
    all_results[year] = {}

    # Skip if grand ensemble already exists
    ensemble_path = OUTPUT_DIR / f"fields_grand_ensemble_{year}.gpkg"
    if ensemble_path.exists():
        print(f"  Grand ensemble exists: {ensemble_path}, loading cached results.")
        for source, engines in SOURCE_ENGINE_MAP.items():
            yr_min, yr_max = SOURCE_YEAR_RANGE[source]
            if year < yr_min or year > yr_max:
                continue
            for engine in engines:
                if engine == "ftw":
                    for m in FTW_MODELS:
                        p = OUTPUT_DIR / f"fields_{source}_{engine}_{m}_{year}.gpkg"
                        if p.exists():
                            all_results[year][f"{source}/ftw/{m}"] = gpd.read_file(p)
                elif engine == "delineate-anything":
                    for m in DA_MODELS:
                        p = OUTPUT_DIR / f"fields_{source}_{engine}_{m}_{year}.gpkg"
                        if p.exists():
                            all_results[year][f"{source}/da/{m}"] = gpd.read_file(p)
                else:
                    p = OUTPUT_DIR / f"fields_{source}_{engine}_{year}.gpkg"
                    if p.exists():
                        all_results[year][f"{source}/{engine}"] = gpd.read_file(p)
        continue

    for source, engines in SOURCE_ENGINE_MAP.items():
        yr_min, yr_max = SOURCE_YEAR_RANGE[source]
        if year < yr_min or year > yr_max:
            continue

        for engine in engines:
            if engine == "ftw":
                for ftw_model in FTW_MODELS:
                    tag = f"{source}/ftw/{ftw_model}"
                    print(f"  {tag}: starting...", flush=True)
                    try:
                        gdf, _ = run_delineation(source, engine, year, model=ftw_model)
                        all_results[year][tag] = gdf
                        print(f"  {tag}: {len(gdf)} fields")
                    except Exception as exc:
                        print(f"  {tag}: FAILED — {exc}")
            elif engine == "delineate-anything":
                for da_model in DA_MODELS:
                    tag = f"{source}/da/{da_model}"
                    print(f"  {tag}: starting...", flush=True)
                    try:
                        gdf, _ = run_delineation(source, engine, year, model=da_model)
                        all_results[year][tag] = gdf
                        print(f"  {tag}: {len(gdf)} fields")
                    except Exception as exc:
                        print(f"  {tag}: FAILED — {exc}")
            else:
                tag = f"{source}/{engine}"
                print(f"  {tag}: starting...", flush=True)
                try:
                    gdf, _ = run_delineation(source, engine, year)
                    all_results[year][tag] = gdf
                    print(f"  {tag}: {len(gdf)} fields")
                except Exception as exc:
                    print(f"  {tag}: FAILED — {exc}")

## Phase 2: Grand Ensemble (Vote Merge) + SAM2 Refinement

Merge all source–engine results per year via majority-vote rasterization, then refine boundaries with SAM2.

In [ ]:
from agribound.engines.ensemble import EnsembleEngine
from agribound.postprocess import filter_polygons

ensemble_results = {}

for year in YEARS:
    year_results = all_results.get(year, {})
    if len(year_results) < 2:
        print(f"{year}: only {len(year_results)} result(s), skipping ensemble.")
        continue

    output_path = OUTPUT_DIR / f"fields_grand_ensemble_{year}.gpkg"

    if output_path.exists():
        print(f"{year}: already exists, loading.")
        ensemble_results[year] = gpd.read_file(output_path)
        continue

    print(f"{year}: merging {len(year_results)} source–engine results...", end=" ")
    try:
        gdf = EnsembleEngine._merge_vote(year_results, VOTE_THRESHOLD)
        gdf = filter_polygons(gdf, min_area_m2=2500)
        print(f"{len(gdf)} fields")

        # SAM2 boundary refinement on the final ensemble
        if SAM_REFINE:
            print(f"  {year}: refining {len(gdf)} ensemble boundaries with SAM2...", flush=True)
            try:
                from agribound.config import AgriboundConfig
                from agribound.engines.samgeo_engine import refine_boundaries

                raster_cache = OUTPUT_DIR / ".agribound_cache"
                raster_candidates = sorted(raster_cache.glob(f"*sentinel2*{year}*.tif"))
                if not raster_candidates:
                    raster_candidates = sorted(raster_cache.glob(f"*{year}*.tif"))
                if raster_candidates:
                    sam_config = AgriboundConfig(
                        source="sentinel2",
                        engine="ftw",
                        year=year,
                        study_area=study_area,
                        output_path=str(output_path),
                        engine_params={
                            "sam_model": SAM_MODEL,
                            "sam_batch_size": SAM_BATCH_SIZE,
                        },
                        device="auto",
                    )
                    gdf = refine_boundaries(gdf, str(raster_candidates[0]), sam_config)
                    print(f"  {year}: SAM2 refined {len(gdf)} boundaries")
                else:
                    print(f"  {year}: no raster found for SAM2 refinement, skipping")
            except Exception as exc:
                print(f"  {year}: SAM2 refinement failed: {exc}")

        # Reproject to match NMOSE reference CRS
        if gdf.crs is not None and str(gdf.crs) != OUTPUT_CRS:
            gdf = gdf.to_crs(OUTPUT_CRS)
        gdf.to_file(output_path, driver="GPKG")
        ensemble_results[year] = gdf
    except Exception as exc:
        print(f"FAILED — {exc}")

## Phase 3: Evaluation Against NMOSE Reference (Lea County)

In [ ]:
print(f"{'Year':<6} {'Source/Engine':<40} {'Fields':>6} {'F1':>6} {'IoU':>6} {'P':>6} {'R':>6}")
print(f"{'-' * 6} {'-' * 40} {'-' * 6} {'-' * 6} {'-' * 6} {'-' * 6} {'-' * 6}")

for year in YEARS:
    for tag, gdf in sorted(all_results.get(year, {}).items()):
        try:
            m = evaluate(gdf, ref_gdf)
            print(
                f"{year:<6} {tag:<40} {len(gdf):>6} "
                f"{m['f1']:.3f} {m['iou_mean']:.3f} "
                f"{m['precision']:.3f} {m['recall']:.3f}"
            )
        except Exception:
            pass

    gdf = ensemble_results.get(year)
    if gdf is not None:
        try:
            m = evaluate(gdf, ref_gdf)
            print(
                f"{year:<6} {'** GRAND ENSEMBLE **':<40} {len(gdf):>6} "
                f"{m['f1']:.3f} {m['iou_mean']:.3f} "
                f"{m['precision']:.3f} {m['recall']:.3f}"
            )
        except Exception:
            pass

## Visualization: Grand Ensemble vs NMOSE Reference

In [ ]:
from agribound.visualize import show_comparison

if ensemble_results:
    latest_year = max(ensemble_results.keys())
    m = show_comparison(
        [ensemble_results[latest_year], ref_gdf],
        labels=[f"Grand Ensemble ({latest_year})", "NMOSE Reference"],
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_ensemble_vs_reference.html"),
    )
    m

## Visualization: Source–Engine Comparison

In [ ]:
if ensemble_results:
    latest = all_results.get(latest_year, {})
    if latest:
        comp_gdfs = list(latest.values())
        comp_labels = list(latest.keys())
        comp_gdfs.append(ensemble_results[latest_year])
        comp_labels.append("Grand Ensemble")

        m_engines = show_comparison(
            comp_gdfs,
            labels=comp_labels,
            basemap="Esri.WorldImagery",
            output_html=str(OUTPUT_DIR / "map_source_engine_comparison.html"),
        )
        m_engines

## Visualization: Ensemble Time Series (2020–2022)

In [ ]:
ts_gdfs = [ensemble_results[y] for y in sorted(ensemble_results.keys())]
ts_labels = [str(y) for y in sorted(ensemble_results.keys())]

if len(ts_gdfs) >= 2:
    m_ts = show_comparison(
        ts_gdfs,
        labels=ts_labels,
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_ensemble_timeseries.html"),
    )
    m_ts